# Session 15: Python Scripts — Execution & Imports

## Learning Objectives

By the end of this session, you will be able to:

1. Understand the difference between running a script and importing from it
2. Import functions and classes from `.py` files
3. Use `if __name__ == "__main__"` to control execution behavior
4. Pass command-line arguments to scripts using `sys.argv`

## Table of Contents

1. [Quick Recap: Running Scripts](#1.-Quick-Recap:-Running-Scripts)
2. [Writing Modular Scripts](#2.-Writing-Modular-Scripts)
3. [Importing from Scripts](#3.-Importing-from-Scripts)
4. [The Problem: Code Runs on Import](#4.-The-Problem:-Code-Runs-on-Import)
5. [The `__name__` Variable](#5.-The-__name__-Variable)
6. [Command-Line Arguments with `sys.argv`](#6.-Command-Line-Arguments-with-sys.argv)
7. [Exercises](#7.-Exercises)
8. [Summary](#8.-Summary)

---

## 1. Quick Recap: Running Scripts

In Session 13, we learned that running `python script.py` in the terminal executes a `.py` file **from top to bottom**. The script runs, produces output, and then Python exits.

Let's start with a simple example. The file `scripts/hello.py` contains three `print()` statements:

In [ ]:
!python scripts/hello.py

Simple enough. But what happens when a script contains **functions, classes, AND executable code**? That's where things get interesting.

---

## 2. Writing Modular Scripts

Real-world scripts usually contain **reusable code** (functions and classes) alongside **demo or test code** that shows how to use them.

Let's look at `scripts/utils.py` — a temperature conversion module:

```python
# utils.py — Temperature conversion utilities (no __name__ guard)


def celsius_to_fahrenheit(celsius):
    """Convert Celsius to Fahrenheit."""
    return celsius * 9 / 5 + 32


def fahrenheit_to_celsius(fahrenheit):
    """Convert Fahrenheit to Celsius."""
    return (fahrenheit - 32) * 5 / 9


def format_temperature(value, unit="C"):
    """Format a temperature value with its unit symbol."""
    return f"{value:.1f} °{unit}"


# Demo code — runs every time this file is executed OR imported
print("=== Temperature Converter ===")
temp_c = 100
temp_f = celsius_to_fahrenheit(temp_c)
print(f"{format_temperature(temp_c, 'C')} = {format_temperature(temp_f, 'F')}")

temp_f = 72
temp_c = fahrenheit_to_celsius(temp_f)
print(f"{format_temperature(temp_f, 'F')} = {format_temperature(temp_c, 'C')}")
```

This file has two parts:
- **Top**: Three reusable functions (`celsius_to_fahrenheit`, `fahrenheit_to_celsius`, `format_temperature`)
- **Bottom**: Demo code that calls those functions and prints results

When we run it directly, everything executes top to bottom:

In [ ]:
!python scripts/utils.py

Now let's look at `scripts/employee.py` — a module that defines a class with demo code:

```python
# employee.py — Employee class module (no __name__ guard)

from datetime import datetime


class Employee:
    """Represents an employee with name, role, and start date."""

    def __init__(self, name, role, start_date):
        self.name = name
        self.role = role
        self.start_date = datetime.strptime(start_date, "%Y-%m-%d")

    def years_employed(self):
        """Calculate years since start date."""
        delta = datetime.now() - self.start_date
        return round(delta.days / 365.25, 1)

    def __repr__(self):
        return f"Employee('{self.name}', '{self.role}')"

    def __str__(self):
        return f"{self.name} — {self.role} ({self.years_employed()} years)"


# Demo code — runs every time this file is executed OR imported
print("--- Employee Demo ---")
emp = Employee("Alice", "Data Analyst", "2022-03-15")
print(emp)
print(f"Years employed: {emp.years_employed()}")
```

Same pattern: a reusable `Employee` class at the top, and demo code at the bottom. Let's run it:

In [ ]:
!python scripts/employee.py

Both scripts work perfectly when run directly. But what if we want to **reuse** those functions and classes in another script or notebook? That's what importing is for.

---

## 3. Importing from Scripts

Python's `import` statement lets you load code from `.py` files and use their functions, classes, and variables in your own code. This is how you **reuse** code without copying and pasting.

Before we import, we need to tell Python where to find our scripts. By default, Python looks in the current directory and standard library paths. Since our scripts are inside `scripts/`, we need to add that folder to the search path:

In [ ]:
import sys

sys.path.insert(0, "scripts")

Now let's import `utils`:

In [ ]:
import utils

Wait — did you see that? The **demo code ran automatically** when we imported the module! We just wanted the functions, but we got all the print statements too.

Let's confirm the functions do work, though:

In [ ]:
result = utils.celsius_to_fahrenheit(25)
print(f"25°C = {result}°F")

The functions work fine — the problem is the unwanted output from the demo code.

### Import Styles

Before we solve that problem, let's review the different ways to import:

| Syntax | What It Does | Example |
|--------|--------------|---------|
| `import X` | Imports the whole module | `import utils` then `utils.celsius_to_fahrenheit(25)` |
| `from X import Y` | Imports specific names | `from utils import celsius_to_fahrenheit` then `celsius_to_fahrenheit(25)` |
| `from X import *` | Imports everything (not recommended) | `from utils import *` then `celsius_to_fahrenheit(25)` |

> **Note:** `from X import *` is generally discouraged because it makes it hard to tell where names came from, and it can overwrite existing variables.

---

## 4. The Problem: Code Runs on Import

When we ran `import utils` above, Python executed the **entire file** — including the demo code at the bottom. This is the fundamental problem:

> **The Problem:** Every time someone imports our module, the demo code runs automatically. This is confusing and unwanted.

The same thing would happen with `employee.py` — importing it would print the demo output.

Think about it from the user's perspective:

- You write a script with useful functions
- You add some demo code at the bottom to test those functions
- A colleague imports your script to use one function
- They get unexpected print statements cluttering their output

We need a way to say: *"Only run this code when the file is executed directly, NOT when it's imported."*

That's exactly what the `__name__` variable does.

---

## 5. The `__name__` Variable

Every Python file has a special built-in variable called `__name__`. Python sets it **automatically** depending on how the file is being used:

- **When run directly** (`python script.py`): `__name__` is set to `"__main__"`
- **When imported** (`import script`): `__name__` is set to the module's name (e.g., `"script"`)

Let's see this in action:

In [ ]:
# In a notebook cell, __name__ is always "__main__"
print(__name__)

In [ ]:
# But when we check an imported module's __name__, it's the module name
print(utils.__name__)

### The Guard Pattern

This difference lets us write a simple `if` statement that controls what runs:

```python
if __name__ == "__main__":
    # This code only runs when the file is executed directly
    # It does NOT run when the file is imported
```

This is called the **`__name__` guard** (or "main guard"). Let's see how `scripts/utils_fixed.py` uses it:

```python
# utils_fixed.py — Temperature conversion utilities (with __name__ guard)


def celsius_to_fahrenheit(celsius):
    """Convert Celsius to Fahrenheit."""
    return celsius * 9 / 5 + 32


def fahrenheit_to_celsius(fahrenheit):
    """Convert Fahrenheit to Celsius."""
    return (fahrenheit - 32) * 5 / 9


def format_temperature(value, unit="C"):
    """Format a temperature value with its unit symbol."""
    return f"{value:.1f} °{unit}"


if __name__ == "__main__":
    # Demo code — only runs when this file is executed directly
    print("=== Temperature Converter ===")
    temp_c = 100
    temp_f = celsius_to_fahrenheit(temp_c)
    print(f"{format_temperature(temp_c, 'C')} = {format_temperature(temp_f, 'F')}")

    temp_f = 72
    temp_c = fahrenheit_to_celsius(temp_f)
    print(f"{format_temperature(temp_f, 'F')} = {format_temperature(temp_c, 'C')}")
```

The functions are **identical** to `utils.py`. The only difference is that the demo code is wrapped in `if __name__ == "__main__":`. Let's import it:

In [ ]:
import utils_fixed

# No output! The demo code is inside the guard.

In [ ]:
# But the functions are still available
result = utils_fixed.celsius_to_fahrenheit(25)
print(f"25°C = {result}°F")

No unwanted output, and the functions work perfectly.

### Behavior Summary

| Scenario | `__name__` value | Guard code runs? |
|----------|-----------------|------------------|
| `python utils_fixed.py` | `"__main__"` | Yes |
| `import utils_fixed` | `"utils_fixed"` | No |
| Run cell in notebook | `"__main__"` | Yes |

Let's verify that running the file directly still works:

In [ ]:
!python scripts/utils_fixed.py

The demo code runs when executed directly, but not when imported. This is exactly what we want.

---

## 6. Command-Line Arguments with `sys.argv`

So far, our scripts have been "hard-coded" — they always do the same thing. But what if we want to pass **input values** to a script when we run it?

That's what **command-line arguments** are for. When you run:

```bash
python greet.py Alice
```

The word `Alice` is a command-line argument. Python makes these available through `sys.argv` — a list of strings containing the script name and any arguments.

Let's try `scripts/greet.py`:

In [ ]:
!python scripts/greet.py Alice

In [ ]:
# What happens if we forget the argument?
!python scripts/greet.py

The script checks `len(sys.argv)` and prints a helpful usage message if the argument is missing. This is good practice.

### How `sys.argv` Works

`sys.argv` is a list where:
- `sys.argv[0]` is always the script name
- `sys.argv[1]`, `sys.argv[2]`, etc. are the arguments you pass

| Command | `sys.argv[0]` | `sys.argv[1]` | `sys.argv[2]` |
|---------|--------------|--------------|---------------|
| `python greet.py Alice` | `"greet.py"` | `"Alice"` | — |
| `python predict.py 5.1 3.5` | `"predict.py"` | `"5.1"` | `"3.5"` |
| `python analyze.py data.csv USD` | `"analyze.py"` | `"data.csv"` | `"USD"` |

> **Important:** All command-line arguments are **strings**. You must convert them explicitly: `int(sys.argv[1])`, `float(sys.argv[1])`.

### Real-World Example: Iris Species Prediction

Here's a more practical script that combines `sys.argv` with machine learning. The file `scripts/predict_species.py` trains a logistic regression model on the iris dataset and predicts a species from four measurements:

```python
# predict_species.py — Predict iris species from flower measurements
#
# Usage:
#   python predict_species.py <sepal_length> <sepal_width> <petal_length> <petal_width>
#
# Example:
#   python predict_species.py 5.1 3.5 1.4 0.2

import sys
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris


def train_model():
    """Train a logistic regression model on the iris dataset."""
    iris = load_iris()
    X = iris.data
    y = iris.target
    model = LogisticRegression(max_iter=1000)
    model.fit(X, y)
    return model, iris.target_names


def predict_species(model, target_names, values):
    """Predict the species for the given feature values."""
    prediction = model.predict(values)
    return target_names[prediction[0]]


if __name__ == "__main__":
    if len(sys.argv) != 5:
        print("Usage: python predict_species.py <sepal_length> <sepal_width> <petal_length> <petal_width>")
        print("Example: python predict_species.py 5.1 3.5 1.4 0.2")
        sys.exit(1)

    # Parse command-line arguments (all strings → convert to floats)
    values = np.array([float(arg) for arg in sys.argv[1:]]).reshape(1, -1)

    # Train and predict
    model, target_names = train_model()
    species = predict_species(model, target_names, values)

    # Display results
    print("Input values:")
    print(f"  sepal_length: {values[0][0]}")
    print(f"  sepal_width:  {values[0][1]}")
    print(f"  petal_length: {values[0][2]}")
    print(f"  petal_width:  {values[0][3]}")
    print("--------------------")
    print(f"Predicted species: {species}")
```

Notice this script uses the `__name__` guard — the `train_model` and `predict_species` functions can be imported without triggering the command-line logic.

Let's try it with two different sets of measurements:

In [ ]:
# Typical setosa measurements (small petals)
!python scripts/predict_species.py 5.1 3.5 1.4 0.2

In [ ]:
# Typical virginica measurements (large petals)
!python scripts/predict_species.py 6.7 3.0 5.2 2.3

Same script, different inputs, different predictions. This is the power of command-line arguments — your script becomes a flexible tool.

---

## 7. Exercises

### Exercise 1: Personalized Greeting

Create a `scripts/my_greet.py` script that takes a **name** and an **age** from the command line and prints a greeting.

**Requirements:**

1. Accept 2 command-line arguments: `name` and `age`
2. Print a greeting using both values
3. Show a usage message if arguments are missing

**Expected usage:**

```bash
python scripts/my_greet.py Alice 25
# Output: Hello, Alice! You are 25 years old.
```

**Hints:**
- `sys.argv[1]` is the name (already a string)
- `sys.argv[2]` is the age (keep it as a string for the print, or convert with `int()` if you want to do math)

In [ ]:
# Your code here: create scripts/my_greet.py
pass

In [ ]:
# Test your script
# !python scripts/my_greet.py Alice 25
# !python scripts/my_greet.py Bob 30

### Exercise 2: Fix the Import Problem

The file `scripts/employee.py` has demo code that runs on import. Your task is to **fix it**.

**Requirements:**

1. Create a new file `scripts/employee_fixed.py` (copy the contents of `employee.py`)
2. Add an `if __name__ == "__main__":` guard around the demo code
3. Verify that:
   - Running `!python scripts/employee_fixed.py` still shows the demo
   - Importing with `from employee_fixed import Employee` produces **no output**

**Hints:**
- Look at how `utils.py` was fixed into `utils_fixed.py` — same pattern
- Only the demo code at the bottom needs to go inside the guard

In [ ]:
# Your code here: create scripts/employee_fixed.py
pass

In [ ]:
# Test 1: Run directly (should show demo output)
# !python scripts/employee_fixed.py

In [ ]:
# Test 2: Import (should have NO demo output)
# from employee_fixed import Employee
#
# emp = Employee("Bob", "Engineer", "2023-01-10")
# print(emp)
# print(f"Years employed: {emp.years_employed()}")

### Exercise 3: Calculator Script

Create a `scripts/calculator.py` script that takes three command-line arguments and performs a basic math operation.

**Requirements:**

1. Accept 3 command-line arguments: `num1`, `operator`, `num2`
2. Support the operators: `+`, `-`, `*`, `/`
3. Print the result in a formatted string
4. Show a usage message if the wrong number of arguments is provided
5. Use the `__name__` guard

**Expected usage:**

```bash
python scripts/calculator.py 10 + 5
# Output: 10.0 + 5.0 = 15.0

python scripts/calculator.py 100 / 3
# Output: 100.0 / 3.0 = 33.333333333333336
```

**Hints:**
- `sys.argv[1]` is the first number (convert with `float()`)
- `sys.argv[2]` is the operator (keep as string)
- `sys.argv[3]` is the second number (convert with `float()`)
- Use `if/elif` to check the operator

In [ ]:
# Your code here: create scripts/calculator.py
pass

In [ ]:
# Test your calculator script
# !python scripts/calculator.py 10 + 5
# !python scripts/calculator.py 100 / 3
# !python scripts/calculator.py 7 '*' 6

---

## 8. Summary

In this session, you learned:

1. **Running scripts**: `python script.py` executes a file top-to-bottom
2. **Importing from scripts**: `import module` or `from module import name` to reuse code
3. **The import problem**: Top-level code runs when a module is imported
4. **The `__name__` guard**: `if __name__ == "__main__":` prevents code from running on import
5. **`sys.argv`**: Access command-line arguments as a list of strings

### Key Takeaways

- Always use `if __name__ == "__main__":` in scripts that contain both reusable code and demo/test code
- `sys.argv` values are always strings — remember to convert types
- Well-structured scripts can serve as both standalone programs AND importable modules

### Next Session Preview

In Session 16, we'll learn how to:
- **Build custom Python libraries** from multiple modules
- **Use `__init__.py`** to define a package's public API
- **Install libraries locally** with `pip install -e .`